In [ ]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio


In [1]:
from fastapi import FastAPI, Query
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import requests
from dotenv import load_dotenv
import os
import spacy
import xml.etree.ElementTree as ET
import json
import requests
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
import os
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
load_dotenv()
import re
import time
from typing import List, Dict, Any, Tuple, Optional
from PatentProvider import PatentProvider

c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [1]:
import os
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict, Tuple, Iterable, Any

import spacy
from fastcoref import spacy_component  # registers "fastcoref" component
from label_studio_sdk import LabelStudio

# =====================================================
# CONFIG
# =====================================================

LABEL_STUDIO_URL = os.getenv("LABEL_STUDIO_URL", "http://localhost:8080")
LABEL_STUDIO_API_KEY = os.getenv("LABEL_STUDIO")
PROJECT_ID = int(os.getenv("LABEL_STUDIO_PROJECT_ID", "4"))

BATCH_SIZE = int(os.getenv("BATCH_SIZE", "2"))
MAX_WORKERS = int(os.getenv("MAX_WORKERS", "2"))  # keep low for stability on GPU

TEXT_FIELD_KEY = os.getenv("TEXT_FIELD_KEY", "text")
TEXT_TO_NAME = os.getenv("TEXT_TO_NAME", "text")

MENTION_FROM_NAME = os.getenv("MENTION_FROM_NAME", "mention")
MENTION_LABEL_VALUE = os.getenv("MENTION_LABEL_VALUE", "MENTION")

REL_FROM_NAME = os.getenv("REL_FROM_NAME", "relations")
COREF_REL_VALUE = os.getenv("COREF_REL_VALUE", "COREF")

DEBUG = os.getenv("DEBUG", "1") == "1"

COREF_MODEL_PATH = os.getenv("COREF_MODEL_PATH", "biu-nlp/lingmess-coref")

CHUNK_CHARS = int(os.getenv("CHUNK_CHARS", "12000"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "600"))

MAX_MENTIONS_PER_TASK = int(os.getenv("MAX_MENTIONS_PER_TASK", "1500"))
MAX_RELATIONS_PER_TASK = int(os.getenv("MAX_RELATIONS_PER_TASK", "5000"))

# =====================================================
# Label Studio client
# =====================================================

if not LABEL_STUDIO_API_KEY:
    raise RuntimeError("Missing LABEL_STUDIO env var (Label Studio API key).")

ls_client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=LABEL_STUDIO_API_KEY)
project = ls_client.projects.get(id=PROJECT_ID)
tasks = list(ls_client.tasks.list(project=project.id))
print("TASKS FOUND:", len(tasks))

create_lock = threading.Lock()

# =====================================================
# spaCy pipeline + fastcoref
# =====================================================

nlp = spacy.load("en_core_web_trf")
nlp.max_length = max(nlp.max_length, CHUNK_CHARS + 50_000)

nlp.add_pipe(
    "fastcoref",
    config={
        "model_architecture": "LingMessCoref",
        "model_path": COREF_MODEL_PATH,
        "device": "cuda",  # set to "cpu" if needed
    },
    last=True,
)

# =====================================================
# Helpers
# =====================================================

def to_dict(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "dict"):
        return obj.dict()
    if hasattr(obj, "__dict__"):
        return dict(obj.__dict__)
    return {}

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def chunk_text(text: str, chunk_chars: int, overlap: int) -> List[Tuple[int, str]]:
    out = []
    i = 0
    n = len(text)
    while i < n:
        j = min(n, i + chunk_chars)
        out.append((i, text[i:j]))
        if j == n:
            break
        i = max(0, j - overlap)
    return out

def make_region_id(i: int) -> str:
    return f"m{i:07d}"

def sanitize_span(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def clusters_to_relations(cluster_ids: List[List[str]]) -> List[Dict]:
    rels = []
    for c in cluster_ids:
        if len(c) < 2:
            continue
        for i in range(len(c) - 1):
            rels.append({
                "from_id": c[i],
                "to_id": c[i + 1],
                "type": "relation",
                "from_name": REL_FROM_NAME,
                "to_name": TEXT_TO_NAME,
                "value": {"relation": COREF_REL_VALUE},
            })
    return rels

def stitch_clusters_by_overlap(clusters: List[List[str]]) -> List[List[str]]:
    parent = {}
    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for c in clusters:
        if len(c) < 2:
            continue
        base = c[0]
        for x in c[1:]:
            union(base, x)

    groups = {}
    for c in clusters:
        for x in c:
            r = find(x)
            groups.setdefault(r, set()).add(x)

    return [sorted(list(s)) for s in groups.values() if len(s) >= 2]

# =====================================================
# Robust coref + fastcoref output adapters
# =====================================================

def _sanitize_text(s: str) -> str:
    s = s.replace("\x00", " ")
    s = re.sub(r"[ \t]+", " ", s)
    return s

def _split_midpoint_on_whitespace(s: str) -> Tuple[str, str]:
    mid = len(s) // 2
    left = s.rfind(" ", 0, mid)
    right = s.find(" ", mid)
    if left == -1 and right == -1:
        return s[:mid], s[mid:]
    if left == -1:
        cut = right
    elif right == -1:
        cut = left
    else:
        cut = left if (mid - left) <= (right - mid) else right
    return s[:cut], s[cut:]

def _get_coref_clusters(doc) -> Any:
    """
    fastcoref version differences:
    - doc._.coref_clusters
    - doc._.coref_clusters_ (some builds)
    """
    if hasattr(doc._, "coref_clusters") and doc._.coref_clusters is not None:
        return doc._.coref_clusters
    if hasattr(doc._, "coref_clusters_") and doc._.coref_clusters_ is not None:
        return doc._.coref_clusters_
    return None

def _iter_mentions_from_cluster(cluster: Any) -> Iterable:
    """
    Handle different shapes:
    - cluster has .mentions
    - cluster itself is a list/tuple of mentions
    - cluster mention is a spaCy Span
    """
    if cluster is None:
        return []
    if hasattr(cluster, "mentions"):
        return cluster.mentions
    if isinstance(cluster, (list, tuple)):
        return cluster
    return []

def _span_offsets(m, chunk_text_str: str):
    """
    Convert mention object to (start_char, end_char).
    Mention can be:
    - spaCy Span => has start_char/end_char
    - tuple/list like (start,end) or (start,end,text)
    - dict with start/end
    """
    # spaCy Span
    if hasattr(m, "start_char") and hasattr(m, "end_char"):
        return int(m.start_char), int(m.end_char)

    # tuple/list
    if isinstance(m, (list, tuple)) and len(m) >= 2:
        try:
            s = int(m[0]); e = int(m[1])
            return s, e
        except Exception:
            return None

    # dict
    if isinstance(m, dict) and "start" in m and "end" in m:
        try:
            s = int(m["start"]); e = int(m["end"])
            return s, e
        except Exception:
            return None

    return None

def run_coref_on_chunk_safe(chunk_text_str: str, depth: int = 0, max_depth: int = 6):
    """
    Returns clusters as lists of (start,end,text) in CHUNK OFFSETS.
    Compatible with multiple fastcoref output shapes.
    """
    chunk_text_str = _sanitize_text(chunk_text_str)
    if not chunk_text_str.strip():
        return []

    # stop splitting when small
    if depth >= max_depth or len(chunk_text_str) < 1500:
        try:
            doc = nlp(chunk_text_str)
        except Exception:
            return []
        clusters = _get_coref_clusters(doc) or []
        return _clusters_to_spans(clusters, chunk_text_str)

    try:
        doc = nlp(chunk_text_str)
        clusters = _get_coref_clusters(doc) or []
        return _clusters_to_spans(clusters, chunk_text_str)

    except IndexError:
        a, b = _split_midpoint_on_whitespace(chunk_text_str)
        return run_coref_on_chunk_safe(a, depth + 1, max_depth) + run_coref_on_chunk_safe(b, depth + 1, max_depth)

    except Exception:
        a, b = _split_midpoint_on_whitespace(chunk_text_str)
        return run_coref_on_chunk_safe(a, depth + 1, max_depth) + run_coref_on_chunk_safe(b, depth + 1, max_depth)

def _clusters_to_spans(clusters: Any, chunk_text_str: str):
    out_clusters = []
    if not clusters:
        return out_clusters

    for cl in clusters:
        spans = []
        for m in _iter_mentions_from_cluster(cl):
            off = _span_offsets(m, chunk_text_str)
            if off is None:
                continue
            s, e = off
            if 0 <= s < e <= len(chunk_text_str):
                spans.append((s, e, chunk_text_str[s:e]))
        if len(spans) >= 2:
            out_clusters.append(spans)
    return out_clusters

# =====================================================
# Task processing
# =====================================================

def process_task(task_id: int):
    task_obj = ls_client.tasks.get(id=task_id)
    task_dict = to_dict(task_obj)

    text = (task_dict.get("data") or {}).get(TEXT_FIELD_KEY, "")
    if not isinstance(text, str) or not text.strip():
        print(f"SKIP task {task_id}: no text")
        return

    if DEBUG:
        print(f"[DEBUG] TASK {task_id}: text len={len(text)}")

    chunks = chunk_text(text, CHUNK_CHARS, CHUNK_OVERLAP)

    all_mentions: List[Dict] = []
    all_clusters_global: List[List[str]] = []

    span2id: Dict[Tuple[int, int], str] = {}
    mention_count = 0

    for base, chunk in chunks:
        if not chunk.strip():
            continue

        try:
            clusters = run_coref_on_chunk_safe(chunk)
        except Exception as e:
            print(f"ERROR coref chunk base={base} task={task_id} -> {e!r}")
            continue

        for cl in clusters:
            cluster_ids = []
            for (s, e, _surface) in cl:
                gs = base + s
                ge = base + e
                if ge > len(text):
                    continue

                key = (gs, ge)
                rid = span2id.get(key)
                if rid is None:
                    if mention_count >= MAX_MENTIONS_PER_TASK:
                        continue
                    rid = make_region_id(mention_count)
                    mention_count += 1
                    span2id[key] = rid

                    all_mentions.append({
                        "id": rid,
                        "type": "labels",
                        "from_name": MENTION_FROM_NAME,
                        "to_name": TEXT_TO_NAME,
                        "value": {
                            "start": gs,
                            "end": ge,
                            "text": sanitize_span(text[gs:ge]),
                            "labels": [MENTION_LABEL_VALUE],
                        },
                    })

                cluster_ids.append(rid)

            # cluster if >=2 unique
            uniq = []
            seen = set()
            for rid in cluster_ids:
                if rid not in seen:
                    uniq.append(rid)
                    seen.add(rid)
            if len(uniq) >= 2:
                all_clusters_global.append(uniq)

    if not all_mentions or len(all_mentions) < 2:
        print(f"SKIP task {task_id}: no mentions produced")
        return

    # dedupe identical clusters
    norm_clusters = []
    seen_c = set()
    for c in all_clusters_global:
        t = tuple(c)
        if t in seen_c:
            continue
        seen_c.add(t)
        norm_clusters.append(c)

    # stitch across overlaps
    norm_clusters = stitch_clusters_by_overlap(norm_clusters)

    relations = clusters_to_relations(norm_clusters)
    if len(relations) > MAX_RELATIONS_PER_TASK:
        relations = relations[:MAX_RELATIONS_PER_TASK]

    result = all_mentions + relations

    with create_lock:
        ls_client.predictions.create(
            task=task_id,
            model_version=f"spacy-fastcoref:{COREF_MODEL_PATH}",
            score=1.0,
            result=result,
        )

    print(
        f"CREATED prediction task {task_id}: "
        f"mentions={len(all_mentions)} clusters={len(norm_clusters)} relations={len(relations)}"
    )

def process_batch(task_batch):
    for task in task_batch:
        tid = getattr(task, "id", None)
        if tid is None:
            continue
        try:
            process_task(tid)
        except Exception as e:
            print("ERROR task", tid, "->", repr(e))

# =====================================================
# RUN
# =====================================================

batches = list(chunked(tasks, BATCH_SIZE))
print("BATCHES:", len(batches), "| BATCH_SIZE:", BATCH_SIZE, "| MAX_WORKERS:", MAX_WORKERS)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(process_batch, b) for b in batches]
    for fut in as_completed(futures):
        fut.result()


c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(
c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
12/20/2025 22:43:29 - INFO - 	 HTTP Request: POST http://localhost:8080/api/token/refresh/ "HTTP/1.1 200 OK"
12/20/2025 22:43:29 - INFO - 	 HTTP Request: GET http://localhost:8080/api/projects/4/ "HTTP/1.1 200 OK"
12/20/2025 22:43:29 - INFO - 	 HTTP Request: GET http://localhost:8080/api/tasks/?fields=all&page=1&project=4 "HTTP/1.1 200 OK"
12/20/2025 22:43:29 - INFO - 	 HTTP Request: GET http://localhost:8080/api/tasks/?fields=all&page=2&project=4 "HTTP/1.1 404 Not Found"


TASKS FOUND: 11


12/20/2025 22:44:53 - INFO - 	 missing_keys: []
12/20/2025 22:44:53 - INFO - 	 unexpected_keys: []
12/20/2025 22:44:53 - INFO - 	 mismatched_keys: []
12/20/2025 22:44:53 - INFO - 	 error_msgs: []
12/20/2025 22:44:53 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M


BATCHES: 6 | BATCH_SIZE: 2 | MAX_WORKERS: 2


12/20/2025 22:44:55 - INFO - 	 HTTP Request: GET http://localhost:8080/api/tasks/27026/ "HTTP/1.1 200 OK"
c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='annotations', input_value=[], input_type=list])
  return self.__pydantic_serializer__.to_python(
12/20/2025 22:44:55 - INFO - 	 HTTP Request: GET http://localhost:8080/api/tasks/27024/ "HTTP/1.1 200 OK"


...

[DEBUG] TASK 27033: text len=34424


Inference: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]
12/20/2025 22:46:27 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 44.78 examples/s]
12/20/2025 22:46:28 - INFO - 	 ***** Running Inference on 1 texts *****
Map: 100%|██████████| 1/1 [00:00<00:00, 45.45 examples/s]
12/20/2025 22:46:31 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.73it/s]
12/20/2025 22:46:32 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 41.19 examples/s]
12/20/2025 22:46:33 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]
12/20/2025 22:46:34 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 55.07 examples/s]
12/20/2025 22:46:36 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]
12/20/2025 22:46:37 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 53.0

CREATED prediction task 27033: mentions=574 clusters=151 relations=423


Inference: 100%|██████████| 1/1 [00:00<00:00,  1.54it/s]
12/20/2025 22:46:43 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 67.61 examples/s]
12/20/2025 22:46:45 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]
12/20/2025 22:46:46 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 46.11 examples/s]
12/20/2025 22:46:48 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.43it/s]
12/20/2025 22:46:50 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 49.01 examples/s]
12/20/2025 22:46:51 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]
12/20/2025 22:46:53 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 46.22 examples/s]
12/20/2025 22:46:55 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00

CREATED prediction task 27034: mentions=1500 clusters=381 relations=1119
